# Legacy RapidDock-Inspired Distance-Biased Transformer Demo

This notebook is a self-contained exploratory prototype, kept as a historical reference rather than the main thesis workflow.

It demonstrates a RapidDock-style molecular docking pipeline:
- 3D conformer generation and distance matrix construction with RDKit
- Distance-biased self-attention in PyTorch
- End-to-end regression for binding affinity
- Comparison to a simple GCN baseline

Use `rapid_dock_paper_aligned_prototype.ipynb` and the Phase 3 notebooks for the thesis-aligned workflow.

In [ ]:
# Section 1: Setup and Library Imports
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from rdkit import Chem
from rdkit.Chem import AllChem
sns.set_theme(style="whitegrid")
print("Libraries loaded.")

In [ ]:
# Section 2: Load and Inspect Data (Real Dataset)
coding_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "data").exists()),
    Path.cwd(),
)
data_path = coding_root / "data/raw/chembl/chembl_binding_affinity_with_smiles.csv"
max_samples = 120  # keep runtime reasonable for 3D conformer generation

try:
    real_data = pd.read_csv(data_path)
    required_cols = {"smiles", "pchembl_value"}
    if not required_cols.issubset(set(real_data.columns)):
        raise ValueError(f"Missing columns: {required_cols - set(real_data.columns)}")

    sample_data = (
        real_data[["smiles", "pchembl_value"]]
        .rename(columns={"pchembl_value": "pchembl"})
        .dropna(subset=["smiles", "pchembl"])
)
    sample_data["pchembl"] = pd.to_numeric(sample_data["pchembl"], errors="coerce")
    sample_data = sample_data.dropna(subset=["pchembl"])
    sample_data = sample_data.sample(
        n=min(max_samples, len(sample_data)), random_state=42
    ).reset_index(drop=True)

    print(f"Loaded real dataset from: {data_path}")
except Exception as e:
    print(f"Falling back to synthetic sample because: {e}")
    sample_data = pd.DataFrame({
        "smiles": [
            "CCO",
            "CC(=O)O",
            "CCN(CC)CC",
            "c1ccccc1",
            "CC(C)O",
            "CCOC(=O)C",
            "CC(C)CC",
            "CC(C)C(=O)O",
            "CC(C)CO",
            "CC(C)C",
        ],
        "pchembl": np.random.uniform(5.0, 8.0, 10),
    })

print("Loaded data preview:")
display(sample_data.head())
sample_data.info()
sample_data.describe()

In [ ]:
# Section 3: Data Cleaning and Preprocessing
# Remove invalid SMILES, generate 3D conformers, compute distance matrices

def mol_to_tensors_3d(smi, max_atoms=30, n_confs=5):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, randomSeed=42)
    if mol.GetNumConformers() == 0:
        return None
    AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=200)
    mol = Chem.RemoveHs(mol)
    n = mol.GetNumAtoms()
    if n > max_atoms:
        return None
    # Atom features: atomic number only (demo)
    AF = np.zeros((max_atoms, 1))
    for i, atom in enumerate(mol.GetAtoms()):
        AF[i, 0] = atom.GetAtomicNum()
    # Average 3D distance matrix
    D_avg = np.zeros((max_atoms, max_atoms))
    for conf in mol.GetConformers():
        pos = conf.GetPositions()[:n]
        dist = np.linalg.norm(pos[:, None, :] - pos[None, :, :], axis=-1)
        D_avg[:n, :n] += dist
    D_avg[:n, :n] /= mol.GetNumConformers()
    mask = np.zeros(max_atoms, dtype=bool)
    mask[:n] = True
    return AF, D_avg, mask

# Apply to all molecules
mol_tensors = []
skipped = 0
for smi in sample_data['smiles']:
    res = mol_to_tensors_3d(smi)
    if res is None:
        skipped += 1
        mol_tensors.append((None, None, None))
    else:
        mol_tensors.append(res)
print(f"Generated 3D tensors for {len(sample_data)-skipped}/{len(sample_data)} molecules.")

In [ ]:
# Section 4: Data Analysis and Aggregation
# Visualize distance matrix for the first valid molecule
valid_example = next((x for x in mol_tensors if x[0] is not None), None)
if valid_example is None:
    raise ValueError("No valid molecule tensors found for visualization.")

AF, D, mask = valid_example
n_valid = int(np.sum(mask))
plt.figure(figsize=(6, 5))
sns.heatmap(D[:n_valid, :n_valid], cmap="viridis", annot=False)
plt.title("3D Distance Matrix (First Valid Molecule)")
plt.xlabel("Atom Index")
plt.ylabel("Atom Index")
plt.show()

# Summary statistics for pChEMBL
print("pChEMBL summary:")
print(sample_data["pchembl"].describe())

In [ ]:
# Section 5: Data Visualization
# Plot pChEMBL distribution
plt.figure(figsize=(6, 4))
sns.histplot(sample_data["pchembl"], bins=10, kde=True)
plt.title("pChEMBL Distribution (Real Sample)")
plt.xlabel("pChEMBL")
plt.ylabel("Count")
plt.show()

# Visualize atom features for the first valid molecule
plt.figure(figsize=(6, 2))
plt.bar(np.arange(AF.shape[0]), AF[:, 0], color="steelblue")
plt.title("Atom Features (Atomic Number) — First Valid Molecule")
plt.xlabel("Atom Index")
plt.ylabel("Atomic Number")
plt.show()

# Section 6: RapidDock-Style Distance-Biased Transformer

This section implements a RapidDock-inspired distance-biased transformer for molecular docking regression.

- Custom self-attention layer with distance bias
- Transformer encoder for molecular graphs
- End-to-end regression for binding affinity (pChEMBL)

All code is pure PyTorch. The model uses the 3D distance matrix as a bias in the attention mechanism.

In [ ]:
# Section 6.1: Distance-Biased Self-Attention Layer (RapidDock style)
import torch.nn.functional as F

class DistanceBiasedAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, bias_scale=1.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.bias_scale = bias_scale
    
    def forward(self, x, dist_matrix, mask):
        # x: [batch, n_atoms, embed_dim]
        # dist_matrix: [batch, n_atoms, n_atoms]
        # mask: [batch, n_atoms]
        B, N, _ = x.shape
        Q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)  # [B, n_heads, N, head_dim]
        K = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # [B, n_heads, N, N]
        # Distance bias: negative distances (closer = higher attention)
        dist_bias = -self.bias_scale * dist_matrix.unsqueeze(1)  # [B, 1, N, N]
        attn_scores = attn_scores + dist_bias
        # Mask invalid atoms
        mask_ = mask.unsqueeze(1).unsqueeze(2)  # [B, 1, 1, N]
        attn_scores = attn_scores.masked_fill(~mask_, float('-inf'))
        attn_weights = F.softmax(attn_scores, dim=-1)
        out = torch.matmul(attn_weights, V)  # [B, n_heads, N, head_dim]
        out = out.transpose(1, 2).contiguous().view(B, N, self.embed_dim)
        out = self.out_proj(out)
        return out

In [ ]:
# Section 6.2: Transformer Encoder for Molecules
class MolTransformerEncoder(nn.Module):
    def __init__(self, embed_dim=64, n_heads=4, n_layers=2, bias_scale=1.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.input_proj = nn.Linear(1, embed_dim)  # Atom features: atomic number only
        self.layers = nn.ModuleList([
            DistanceBiasedAttention(embed_dim, n_heads, bias_scale) for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(n_layers)])
        self.readout = nn.Linear(embed_dim, 1)  # Regression output
    
    def forward(self, atom_features, dist_matrix, mask):
        # atom_features: [batch, n_atoms, 1]
        # dist_matrix: [batch, n_atoms, n_atoms]
        # mask: [batch, n_atoms]
        x = self.input_proj(atom_features)
        for attn, norm in zip(self.layers, self.norms):
            x = attn(x, dist_matrix, mask)
            x = norm(x)
        # Readout: masked mean pooling
        mask = mask.unsqueeze(-1)
        x = x * mask
        mol_vec = x.sum(dim=1) / mask.sum(dim=1)
        out = self.readout(mol_vec)
        return out.squeeze(-1)

In [ ]:
# Section 6.3: Prepare Data for Model Training
def collate_mol_batch(mol_tensors, pchembl):
    batch_AF = []
    batch_D = []
    batch_mask = []
    batch_y = []
    for i, (AF, D, mask) in enumerate(mol_tensors):
        if AF is None:
            continue
        batch_AF.append(torch.tensor(AF, dtype=torch.float32))
        batch_D.append(torch.tensor(D, dtype=torch.float32))
        batch_mask.append(torch.tensor(mask, dtype=torch.bool))
        batch_y.append(torch.tensor(pchembl[i], dtype=torch.float32))
    AF = torch.stack(batch_AF)
    D = torch.stack(batch_D)
    mask = torch.stack(batch_mask)
    y = torch.stack(batch_y)
    return AF, D, mask, y

AF, D, mask, y = collate_mol_batch(mol_tensors, sample_data['pchembl'].values)
print(f"Batch shapes: AF {AF.shape}, D {D.shape}, mask {mask.shape}, y {y.shape}")

In [ ]:
# Section 6.4: Training Loop for RapidDock-Style Transformer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MolTransformerEncoder(embed_dim=64, n_heads=4, n_layers=2, bias_scale=1.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

AF, D, mask, y = AF.to(device), D.to(device), mask.to(device), y.to(device)
epochs = 50
losses = []
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(AF, D, mask)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(losses, marker="o")
plt.title("Training Loss (RapidDock-Style Transformer)")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.show()

In [ ]:
# Section 6.5: Model Evaluation and Prediction
model.eval()
with torch.no_grad():
    pred = model(AF, D, mask)
    mse = loss_fn(pred, y).item()
    print(f"Final MSE: {mse:.4f}")
    plt.figure(figsize=(6, 4))
    plt.scatter(y.cpu().numpy(), pred.cpu().numpy(), color="teal", s=60)
    plt.plot([y.min().item(), y.max().item()], [y.min().item(), y.max().item()], 'r--')
    plt.xlabel("True pChEMBL")
    plt.ylabel("Predicted pChEMBL")
    plt.title("RapidDock-Style Transformer: Prediction vs. True")
    plt.show()

# Section 7: GCN Baseline and Side-by-Side Comparison

This section adds a simple molecular GCN baseline and compares it against the RapidDock-style transformer.

- Build adjacency from 3D distances
- Train a 2-layer GCN regressor
- Compare MSE, MAE, and $R^2$ with prediction scatter plots

Note: this remains a compact in-notebook comparison built for demonstration and exploratory reference, not a production evaluation pipeline.

In [ ]:
# Section 7.1: GCN Baseline Model Definition
def build_adjacency_from_distance(D, mask, cutoff=2.2):
    # D: [B, N, N], mask: [B, N]
    B, N, _ = D.shape
    adj = (D <= cutoff).float()
    # Ensure self-loops
    eye = torch.eye(N, device=D.device).unsqueeze(0).expand(B, -1, -1)
    adj = torch.maximum(adj, eye)
    # Mask invalid atoms in rows/cols
    m_row = mask.unsqueeze(-1).float()
    m_col = mask.unsqueeze(1).float()
    adj = adj * m_row * m_col
    return adj

class SimpleGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        # Symmetric normalization: A_hat = D^{-1/2} A D^{-1/2}
        deg = adj.sum(dim=-1) + 1e-6
        deg_inv_sqrt = deg.pow(-0.5)
        A_hat = deg_inv_sqrt.unsqueeze(-1) * adj * deg_inv_sqrt.unsqueeze(1)
        x = torch.bmm(A_hat, x)
        return self.lin(x)

class MolGCNBaseline(nn.Module):
    def __init__(self, in_dim=1, hidden_dim=64):
        super().__init__()
        self.gcn1 = SimpleGCNLayer(in_dim, hidden_dim)
        self.gcn2 = SimpleGCNLayer(hidden_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.readout = nn.Linear(hidden_dim, 1)

    def forward(self, atom_features, adj, mask):
        x = F.relu(self.norm1(self.gcn1(atom_features, adj)))
        x = F.relu(self.norm2(self.gcn2(x, adj)))
        m = mask.unsqueeze(-1).float()
        x = x * m
        mol_vec = x.sum(dim=1) / m.sum(dim=1).clamp_min(1.0)
        return self.readout(mol_vec).squeeze(-1)

In [ ]:
# Section 7.2: Train GCN Baseline
adj = build_adjacency_from_distance(D, mask, cutoff=2.2)
gcn_model = MolGCNBaseline(in_dim=1, hidden_dim=64).to(device)
gcn_opt = torch.optim.Adam(gcn_model.parameters(), lr=1e-3)
gcn_loss_fn = nn.MSELoss()

gcn_losses = []
epochs = 50
for epoch in range(epochs):
    gcn_model.train()
    gcn_opt.zero_grad()
    gcn_pred = gcn_model(AF, adj, mask)
    gcn_loss = gcn_loss_fn(gcn_pred, y)
    gcn_loss.backward()
    gcn_opt.step()
    gcn_losses.append(gcn_loss.item())
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"[GCN] Epoch {epoch+1}/{epochs} | Loss: {gcn_loss.item():.4f}")

plt.figure(figsize=(6, 4))
plt.plot(gcn_losses, marker="o", color="darkorange")
plt.title("Training Loss (GCN Baseline)")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.show()

In [ ]:
# Section 7.3: Compare RapidDock-Style Transformer vs GCN
def regression_metrics(y_true, y_pred):
    y_true = y_true.detach().cpu().numpy()
    y_pred = y_pred.detach().cpu().numpy()
    mse = float(np.mean((y_true - y_pred) ** 2))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2) + 1e-12)
    r2 = 1.0 - ss_res / ss_tot
    return mse, mae, r2

model.eval()
gcn_model.eval()
with torch.no_grad():
    tf_pred = model(AF, D, mask)
    gcn_pred = gcn_model(AF, adj, mask)

tf_mse, tf_mae, tf_r2 = regression_metrics(y, tf_pred)
gcn_mse, gcn_mae, gcn_r2 = regression_metrics(y, gcn_pred)

comparison_df = pd.DataFrame({
    "Model": ["RapidDock-Style Transformer", "GCN Baseline"],
    "MSE": [tf_mse, gcn_mse],
    "MAE": [tf_mae, gcn_mae],
    "R2": [tf_r2, gcn_r2],
})
print("Model Comparison:")
display(comparison_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y.cpu().numpy(), tf_pred.cpu().numpy(), s=65, label="Transformer", color="teal")
axes[0].scatter(y.cpu().numpy(), gcn_pred.cpu().numpy(), s=65, label="GCN", color="darkorange", alpha=0.85)
y_min, y_max = y.min().item(), y.max().item()
axes[0].plot([y_min, y_max], [y_min, y_max], "r--", lw=1.5)
axes[0].set_title("Predicted vs True")
axes[0].set_xlabel("True pChEMBL")
axes[0].set_ylabel("Predicted pChEMBL")
axes[0].legend()

metric_plot_df = comparison_df.melt(id_vars="Model", value_vars=["MSE", "MAE", "R2"],
                                    var_name="Metric", value_name="Value")
sns.barplot(data=metric_plot_df, x="Metric", y="Value", hue="Model", ax=axes[1])
axes[1].set_title("Metric Comparison")
axes[1].set_xlabel("Metric")
axes[1].set_ylabel("Value")
axes[1].legend(title="Model")

plt.tight_layout()
plt.show()

# Section 8: Scaffold Split with Early Stopping

This section extends the legacy demo with a more realistic exploratory evaluation protocol.

- Split molecules into train and validation sets by Murcko scaffold
- Train both models with early stopping on validation loss
- Compare validation metrics (MSE, MAE, and $R^2$)

This remains a compact reference implementation rather than part of the main thesis execution workflow.

In [ ]:
# Section 8.1: Create Scaffold-Based Train/Validation Split
from rdkit.Chem.Scaffolds import MurckoScaffold

torch.manual_seed(42)
np.random.seed(42)

n_samples = AF.shape[0]
val_fraction = 0.2
target_val_size = max(1, int(round(val_fraction * n_samples)))

# Recover the same valid molecule order used to build AF/D/mask/y
valid_src_idx = [i for i, (af_i, _, _) in enumerate(mol_tensors) if af_i is not None]
valid_smiles = sample_data.iloc[valid_src_idx]["smiles"].tolist()
if len(valid_smiles) != n_samples:
    raise ValueError("Mismatch between tensor batch size and valid smiles count.")

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return "INVALID"
    try:
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaf if scaf else "EMPTY"
    except Exception:
        return "INVALID"

scaffold_to_positions = {}
for pos, smi in enumerate(valid_smiles):
    scaf = safe_scaffold(smi)
    scaffold_to_positions.setdefault(scaf, []).append(pos)

# Greedy scaffold split: allocate largest scaffold groups to validation first
scaffold_groups = sorted(scaffold_to_positions.values(), key=len, reverse=True)
val_positions = []
for group in scaffold_groups:
    if len(val_positions) < target_val_size:
        val_positions.extend(group)
    else:
        break

val_positions = sorted(set(val_positions))
train_positions = sorted(set(range(n_samples)) - set(val_positions))

# Safety fallback if scaffold split collapses (very small/imbalanced sample)
if len(train_positions) == 0 or len(val_positions) == 0:
    indices = np.random.permutation(n_samples)
    split = max(1, int(0.8 * n_samples))
    train_positions = sorted(indices[:split].tolist())
    val_positions = sorted(indices[split:].tolist())
    split_mode = "random-fallback"
else:
    split_mode = "scaffold"

train_idx = torch.tensor(train_positions, dtype=torch.long, device=AF.device)
val_idx = torch.tensor(val_positions, dtype=torch.long, device=AF.device)

AF_train, AF_val = AF[train_idx], AF[val_idx]
D_train, D_val = D[train_idx], D[val_idx]
mask_train, mask_val = mask[train_idx], mask[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

adj_all = build_adjacency_from_distance(D, mask, cutoff=2.2)
adj_train, adj_val = adj_all[train_idx], adj_all[val_idx]

train_scaf = {safe_scaffold(valid_smiles[i]) for i in train_positions}
val_scaf = {safe_scaffold(valid_smiles[i]) for i in val_positions}
overlap_scaf = train_scaf.intersection(val_scaf)

print(f"Split mode: {split_mode}")
print(f"Train size: {AF_train.shape[0]} | Val size: {AF_val.shape[0]}")
print(f"Unique scaffolds -> train: {len(train_scaf)}, val: {len(val_scaf)}, overlap: {len(overlap_scaf)}")

In [ ]:
# Section 8.2: Early-Stopping Training and Validation Comparison
def regression_metrics_np(y_true, y_pred):
    mse = float(np.mean((y_true - y_pred) ** 2))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2) + 1e-12)
    r2 = 1.0 - ss_res / ss_tot
    return mse, mae, r2

def train_transformer_with_early_stopping(patience=20, max_epochs=300, lr=1e-3):
    tf_model = MolTransformerEncoder(embed_dim=64, n_heads=4, n_layers=2, bias_scale=1.0).to(device)
    opt = torch.optim.Adam(tf_model.parameters(), lr=lr)
    crit = nn.MSELoss()

    best_state = None
    best_val = float("inf")
    wait = 0
    train_losses, val_losses = [], []

    for epoch in range(max_epochs):
        tf_model.train()
        opt.zero_grad()
        pred_train = tf_model(AF_train, D_train, mask_train)
        loss_train = crit(pred_train, y_train)
        loss_train.backward()
        opt.step()

        tf_model.eval()
        with torch.no_grad():
            pred_val = tf_model(AF_val, D_val, mask_val)
            loss_val = crit(pred_val, y_val)

        train_losses.append(loss_train.item())
        val_losses.append(loss_val.item())

        if loss_val.item() < best_val - 1e-8:
            best_val = loss_val.item()
            best_state = {k: v.detach().clone() for k, v in tf_model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        tf_model.load_state_dict(best_state)

    return tf_model, train_losses, val_losses

def train_gcn_with_early_stopping(patience=20, max_epochs=300, lr=1e-3):
    g_model = MolGCNBaseline(in_dim=1, hidden_dim=64).to(device)
    opt = torch.optim.Adam(g_model.parameters(), lr=lr)
    crit = nn.MSELoss()

    best_state = None
    best_val = float("inf")
    wait = 0
    train_losses, val_losses = [], []

    for epoch in range(max_epochs):
        g_model.train()
        opt.zero_grad()
        pred_train = g_model(AF_train, adj_train, mask_train)
        loss_train = crit(pred_train, y_train)
        loss_train.backward()
        opt.step()

        g_model.eval()
        with torch.no_grad():
            pred_val = g_model(AF_val, adj_val, mask_val)
            loss_val = crit(pred_val, y_val)

        train_losses.append(loss_train.item())
        val_losses.append(loss_val.item())

        if loss_val.item() < best_val - 1e-8:
            best_val = loss_val.item()
            best_state = {k: v.detach().clone() for k, v in g_model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        g_model.load_state_dict(best_state)

    return g_model, train_losses, val_losses

tf_es_model, tf_train_losses, tf_val_losses = train_transformer_with_early_stopping()
gcn_es_model, gcn_train_losses, gcn_val_losses = train_gcn_with_early_stopping()

tf_es_model.eval()
gcn_es_model.eval()
with torch.no_grad():
    tf_val_pred = tf_es_model(AF_val, D_val, mask_val).detach().cpu().numpy()
    gcn_val_pred = gcn_es_model(AF_val, adj_val, mask_val).detach().cpu().numpy()
    y_val_np = y_val.detach().cpu().numpy()

tf_mse_val, tf_mae_val, tf_r2_val = regression_metrics_np(y_val_np, tf_val_pred)
gcn_mse_val, gcn_mae_val, gcn_r2_val = regression_metrics_np(y_val_np, gcn_val_pred)

val_comparison_df = pd.DataFrame({
    "Model": ["RapidDock-Style Transformer", "GCN Baseline"],
    "Val_MSE": [tf_mse_val, gcn_mse_val],
    "Val_MAE": [tf_mae_val, gcn_mae_val],
    "Val_R2": [tf_r2_val, gcn_r2_val],
})
print("Validation Comparison (Early Stopping):")
display(val_comparison_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tf_train_losses, label="Transformer Train", color="teal")
axes[0].plot(tf_val_losses, label="Transformer Val", color="teal", linestyle="--")
axes[0].plot(gcn_train_losses, label="GCN Train", color="darkorange")
axes[0].plot(gcn_val_losses, label="GCN Val", color="darkorange", linestyle="--")
axes[0].set_title("Train vs Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()

axes[1].scatter(y_val_np, tf_val_pred, s=70, color="teal", label="Transformer")
axes[1].scatter(y_val_np, gcn_val_pred, s=70, color="darkorange", label="GCN", alpha=0.85)
line_min = float(min(np.min(y_val_np), np.min(tf_val_pred), np.min(gcn_val_pred)))
line_max = float(max(np.max(y_val_np), np.max(tf_val_pred), np.max(gcn_val_pred)))
axes[1].plot([line_min, line_max], [line_min, line_max], "r--", lw=1.5)
axes[1].set_title("Validation: Predicted vs True")
axes[1].set_xlabel("True pChEMBL")
axes[1].set_ylabel("Predicted pChEMBL")
axes[1].legend()

plt.tight_layout()
plt.show()

# Section 9: Scaffold-Group Cross-Validation

This section adds scaffold-group cross-validation for a more stable exploratory estimate than a single split.

- Build folds from Murcko scaffold groups
- Train Transformer and GCN models with early stopping per fold
- Report fold-level and aggregate metrics (mean $\pm$ std)

Like the rest of this notebook, this section is retained as a legacy reference rather than a thesis-aligned production pipeline.

In [ ]:
# Section 9.1: Run Scaffold-Group Cross-Validation
from collections import defaultdict

if "regression_metrics_np" not in globals():
    def regression_metrics_np(y_true, y_pred):
        mse = float(np.mean((y_true - y_pred) ** 2))
        mae = float(np.mean(np.abs(y_true - y_pred)))
        ss_res = float(np.sum((y_true - y_pred) ** 2))
        ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2) + 1e-12)
        r2 = 1.0 - ss_res / ss_tot
        return mse, mae, r2

# Reconstruct valid smiles aligned with AF/D/mask/y tensors
valid_src_idx = [i for i, (af_i, _, _) in enumerate(mol_tensors) if af_i is not None]
valid_smiles = sample_data.iloc[valid_src_idx]["smiles"].tolist()

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return "INVALID"
    try:
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaf if scaf else "EMPTY"
    except Exception:
        return "INVALID"

scaffold_to_positions = defaultdict(list)
for pos, smi in enumerate(valid_smiles):
    scaffold_to_positions[safe_scaffold(smi)].append(pos)

# Round-robin assign scaffold groups to folds to balance fold sizes
k_folds = 3
groups_sorted = sorted(scaffold_to_positions.values(), key=len, reverse=True)
folds = [[] for _ in range(k_folds)]
fold_sizes = [0] * k_folds
for grp in groups_sorted:
    j = int(np.argmin(fold_sizes))
    folds[j].extend(grp)
    fold_sizes[j] += len(grp)

def train_transformer_fold(AF_tr, D_tr, M_tr, y_tr, AF_va, D_va, M_va, y_va, max_epochs=120, patience=15, lr=1e-3):
    m = MolTransformerEncoder(embed_dim=64, n_heads=4, n_layers=2, bias_scale=1.0).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    crit = nn.MSELoss()
    best_state, best_val, wait = None, float("inf"), 0
    for _ in range(max_epochs):
        m.train()
        opt.zero_grad()
        p_tr = m(AF_tr, D_tr, M_tr)
        l_tr = crit(p_tr, y_tr)
        l_tr.backward()
        opt.step()

        m.eval()
        with torch.no_grad():
            p_va = m(AF_va, D_va, M_va)
            l_va = crit(p_va, y_va)

        if l_va.item() < best_val - 1e-8:
            best_val = l_va.item()
            best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        m.load_state_dict(best_state)
    return m

def train_gcn_fold(AF_tr, A_tr, M_tr, y_tr, AF_va, A_va, M_va, y_va, max_epochs=120, patience=15, lr=1e-3):
    m = MolGCNBaseline(in_dim=1, hidden_dim=64).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    crit = nn.MSELoss()
    best_state, best_val, wait = None, float("inf"), 0
    for _ in range(max_epochs):
        m.train()
        opt.zero_grad()
        p_tr = m(AF_tr, A_tr, M_tr)
        l_tr = crit(p_tr, y_tr)
        l_tr.backward()
        opt.step()

        m.eval()
        with torch.no_grad():
            p_va = m(AF_va, A_va, M_va)
            l_va = crit(p_va, y_va)

        if l_va.item() < best_val - 1e-8:
            best_val = l_va.item()
            best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        m.load_state_dict(best_state)
    return m

adj_all = build_adjacency_from_distance(D, mask, cutoff=2.2)
fold_rows = []
for f in range(k_folds):
    val_pos = sorted(set(folds[f]))
    train_pos = sorted(set(range(AF.shape[0])) - set(val_pos))
    if len(train_pos) == 0 or len(val_pos) == 0:
        continue

    tr_idx = torch.tensor(train_pos, dtype=torch.long, device=AF.device)
    va_idx = torch.tensor(val_pos, dtype=torch.long, device=AF.device)

    AF_tr, D_tr, M_tr, y_tr = AF[tr_idx], D[tr_idx], mask[tr_idx], y[tr_idx]
    AF_va, D_va, M_va, y_va = AF[va_idx], D[va_idx], mask[va_idx], y[va_idx]
    A_tr, A_va = adj_all[tr_idx], adj_all[va_idx]

    tf_model_f = train_transformer_fold(AF_tr, D_tr, M_tr, y_tr, AF_va, D_va, M_va, y_va)
    gcn_model_f = train_gcn_fold(AF_tr, A_tr, M_tr, y_tr, AF_va, A_va, M_va, y_va)

    tf_model_f.eval()
    gcn_model_f.eval()
    with torch.no_grad():
        tf_pred_f = tf_model_f(AF_va, D_va, M_va).detach().cpu().numpy()
        gcn_pred_f = gcn_model_f(AF_va, A_va, M_va).detach().cpu().numpy()
        y_va_np = y_va.detach().cpu().numpy()

    tf_mse_f, tf_mae_f, tf_r2_f = regression_metrics_np(y_va_np, tf_pred_f)
    gcn_mse_f, gcn_mae_f, gcn_r2_f = regression_metrics_np(y_va_np, gcn_pred_f)

    fold_rows.append({"Fold": f + 1, "Model": "RapidDock-Style Transformer", "MSE": tf_mse_f, "MAE": tf_mae_f, "R2": tf_r2_f, "N_val": len(val_pos)})
    fold_rows.append({"Fold": f + 1, "Model": "GCN Baseline", "MSE": gcn_mse_f, "MAE": gcn_mae_f, "R2": gcn_r2_f, "N_val": len(val_pos)})

cv_df = pd.DataFrame(fold_rows)
print("Scaffold-Group CV Fold Results:")
display(cv_df)

cv_summary = (
    cv_df.groupby("Model")[["MSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .round(4)
    .reset_index()
)
print("Scaffold-Group CV Summary (mean ± std):")
display(cv_summary)

plot_df = cv_df.melt(id_vars=["Fold", "Model"], value_vars=["MSE", "MAE", "R2"], var_name="Metric", value_name="Value")
plt.figure(figsize=(12, 4))
sns.boxplot(data=plot_df, x="Metric", y="Value", hue="Model")
plt.title("Scaffold-Group CV Metric Distribution by Fold")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.legend(title="Model")
plt.tight_layout()
plt.show()